# Conveyer · GPU LTX no Colab / Kaggle

Este notebook transforma a GPU do Colab (ou Kaggle) no gerador de clipes LTX do Conveyer.
Ele escolhe o perfil pela VRAM:

| GPU | perfil | modelo | saída |
|---|---|---|---|
| ≥ 40 GB (A100, H100, G4) | **full** | LTX 22B (o mesmo do Runpod), ~37 GB de pesos | 1080p nativo |
| 16 GB (**T4 grátis** do Colab/Kaggle) | **lite** | LTX-Video 2B distilled, ~11,5 GB de pesos | 832×480, ampliado para 1080p |

1. **Ambiente de execução → Alterar tipo → GPUs: T4** (ou melhor, se tiver). No Kaggle: *Settings → Accelerator → GPU T4* e **Internet ligada**.
2. Rode as células **em ordem**. A primeira execução baixa os modelos (perfil lite: ~5–10 min).
3. A célula 6 mostra um **endereço https** e um **token**. Cole os dois no Conveyer em
   `/tools/ltx-scene-test` → *GPU do Colab ou Kaggle* → **Conectar notebook**.
4. Deixe esta aba aberta enquanto o Conveyer gera. Para encerrar, *Ambiente → Desconectar e excluir*.

O ComfyUI escuta só dentro do notebook (127.0.0.1). O túnel expõe apenas um proxy que exige o token
e repassa somente as rotas que o Conveyer usa.

In [ ]:
# 1 · Configuração e checagem da GPU
import os, shutil, subprocess, sys, json, secrets, time, re, hashlib
HF_TOKEN = ""          # opcional: token do Hugging Face, se o download pedir login
VERIFY_SHA256 = True   # confere os hashes dos arquivos baixados; desligue só se já conferiu
PROFILE = ""           # vazio = automático pela VRAM; "lite" força o modelo pequeno

ON_KAGGLE = os.path.exists("/kaggle")
WORK = "/tmp/conveyer" if ON_KAGGLE else "/content/conveyer"
os.makedirs(WORK, exist_ok=True)
COMFY_DIR = f"{WORK}/ComfyUI"
if not HF_TOKEN:
    try:
        if ON_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
        else:
            from google.colab import userdata
            HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = ""

smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                     capture_output=True, text=True)
if smi.returncode != 0:
    raise SystemExit("Nenhuma GPU encontrada. Ative uma GPU (T4) no tipo de ambiente e rode de novo.")
gpu_name, vram_mb = [x.strip() for x in smi.stdout.strip().splitlines()[0].split(",")]
vram_gb = int(vram_mb) / 1024
if not PROFILE:
    PROFILE = "full" if vram_gb >= 40 else "lite"
if vram_gb < 14:
    raise SystemExit(f"{gpu_name} tem só {vram_gb:.0f} GB de VRAM; o perfil lite precisa de uma T4 (16 GB) ou maior.")
need_gb = 45 if PROFILE == "full" else 16
free_gb = shutil.disk_usage(WORK).free / 1024**3
print(f"GPU: {gpu_name} · {vram_gb:.0f} GB VRAM · perfil: {PROFILE} · disco livre: {free_gb:.0f} GB")
if free_gb < need_gb:
    raise SystemExit(f"Espaço em disco insuficiente: o perfil {PROFILE} precisa de ~{need_gb} GB livres.")

In [ ]:
# 2 · ComfyUI fixado no commit validado
COMFY_COMMIT = "3f767e7f67bc587e88d6de6668eb424f725d4649"
if not os.path.isdir(f"{COMFY_DIR}/.git"):
    subprocess.run(["git", "clone", "https://github.com/comfyanonymous/ComfyUI.git", COMFY_DIR], check=True)
subprocess.run(["git", "-C", COMFY_DIR, "fetch", "--quiet", "origin", COMFY_COMMIT], check=False)
subprocess.run(["git", "-C", COMFY_DIR, "checkout", "--quiet", COMFY_COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", f"{COMFY_DIR}/requirements.txt",
                "huggingface_hub", "aiohttp"], check=True)
print("ComfyUI pronto em", COMFY_DIR)

In [ ]:
# 3 · Modelos (cópias de src/lib/services/ltx/models.json e models-lite.json)
ALL_MODELS = json.loads(r'''{
 "full": {
  "profile": "full",
  "files": [
   {
    "path": "diffusion_models/ltx-2.5-22b-distilled-transformer-nvfp4.safetensors",
    "bytes": 18721548408,
    "sha256": "4b94231e734c1950f8f6826cb8bd8715d94be5b3e04f8256ee060c5bc3886c30",
    "repo": "Lightricks/LTX-2.5",
    "revision": "2356ce76915d6c48d313d7e8b25900e1dd3abaa8",
    "file": "diffusion_models/ltx-2.5-22b-distilled-transformer-nvfp4.safetensors"
   },
   {
    "path": "latent_upscale_models/ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors",
    "bytes": 995778752,
    "sha256": "eb5a71fe4068ee87ccdb1c3aa635e547ca76bd2d30ae20ae889f2c325c0677e8",
    "repo": "Lightricks/LTX-2.5",
    "revision": "2356ce76915d6c48d313d7e8b25900e1dd3abaa8",
    "file": "latent_upscale_models/ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors"
   },
   {
    "path": "text_encoders/gemma4-12b-with-proj-ltx-2.5-comfy-int8-convrot.safetensors",
    "bytes": 15372969374,
    "sha256": "6ce688a0aa98a5fa36a9f1e6c3f42152a498cc2b53ee8c15674c64244f91487f",
    "repo": "Lightricks/LTX-2.5",
    "revision": "2356ce76915d6c48d313d7e8b25900e1dd3abaa8",
    "file": "text_encoders/gemma4-12b-with-proj-ltx-2.5-comfy-int8-convrot.safetensors"
   },
   {
    "path": "vae/ltx-2.5-audio-vae-bf16.safetensors",
    "bytes": 364866540,
    "sha256": "c52733d37f6a7fb7949c3dc0fb468c6cb2169e4d836983a73babb9f0d54837a5",
    "repo": "Lightricks/LTX-2.5",
    "revision": "2356ce76915d6c48d313d7e8b25900e1dd3abaa8",
    "file": "vae/ltx-2.5-audio-vae-bf16.safetensors"
   },
   {
    "path": "vae/ltx-2.5-video-vae-conv-bf16.safetensors",
    "bytes": 1452269922,
    "sha256": "685b06ee3d9b2039647698fc4ea33175112462fc374e2777312c907897dfce8d",
    "repo": "Lightricks/LTX-2.5",
    "revision": "2356ce76915d6c48d313d7e8b25900e1dd3abaa8",
    "file": "vae/ltx-2.5-video-vae-conv-bf16.safetensors"
   }
  ]
 },
 "lite": {
  "profile": "lite",
  "files": [
   {
    "repo": "Lightricks/LTX-Video",
    "revision": "8984fa25007f376c1a299016d0957a37a2f797bb",
    "file": "ltxv-2b-0.9.8-distilled.safetensors",
    "path": "checkpoints/ltxv-2b-0.9.8-distilled.safetensors",
    "bytes": 6340744492,
    "sha256": "76aa8c4786af752fa6f951947129d5290c3c6c0b2fadcadea6b5e114ae2cad8f"
   },
   {
    "repo": "comfyanonymous/flux_text_encoders",
    "revision": "6af2a98e3f615bdfa612fbd85da93d1ed5f69ef5",
    "file": "t5xxl_fp8_e4m3fn_scaled.safetensors",
    "path": "text_encoders/t5xxl_fp8_e4m3fn_scaled.safetensors",
    "bytes": 5157348688,
    "sha256": "a498f0485dc9536735258018417c3fd7758dc3bccc0a645feaa472b34955557a"
   }
  ]
 }
}''')
MODELS = ALL_MODELS[PROFILE]
MODELS_JSON = f"{WORK}/models.json"
with open(MODELS_JSON, "w") as fh:
    json.dump(MODELS, fh)
from huggingface_hub import hf_hub_download
models_dir = f"{COMFY_DIR}/models"
for item in MODELS["files"]:
    target = f"{models_dir}/{item['path']}"
    marker = target + ".conveyer-sha256-ok"
    if os.path.isfile(marker) and os.path.getsize(target) == item["bytes"]:
        print("ok (já conferido):", item["path"]); continue
    if not (os.path.isfile(target) and os.path.getsize(target) == item["bytes"]):
        print("baixando:", item["path"], f"({item['bytes'] / 1024**3:.1f} GB)")
        staging = f"{WORK}/hf/{item['repo'].replace('/', '__')}"
        downloaded = hf_hub_download(item["repo"], item["file"], revision=item["revision"], local_dir=staging,
                                     token=HF_TOKEN or None)
        os.makedirs(os.path.dirname(target), exist_ok=True)
        os.replace(downloaded, target)
    if os.path.getsize(target) != item["bytes"]:
        raise SystemExit(f"Tamanho inesperado em {item['path']}")
    if VERIFY_SHA256:
        digest = hashlib.sha256()
        with open(target, "rb") as fh:
            for block in iter(lambda: fh.read(1 << 24), b""):
                digest.update(block)
        if digest.hexdigest() != item["sha256"]:
            os.remove(target)
            raise SystemExit(f"SHA-256 diferente em {item['path']} — arquivo removido; rode a célula de novo.")
    open(marker, "w").close()
    print("ok:", item["path"])

In [ ]:
# 4 · Inicia o ComfyUI (só em 127.0.0.1)
import urllib.request
comfy_log = open(f"{WORK}/comfyui.log", "a")
# Na T4 (16 GB) os pesos não ficam todos na VRAM ao mesmo tempo: --lowvram deixa o ComfyUI alternar.
comfy_args = ["--listen", "127.0.0.1", "--port", "8188"] + (["--lowvram"] if PROFILE == "lite" else [])
comfy = subprocess.Popen([sys.executable, "main.py", *comfy_args],
                         cwd=COMFY_DIR, stdout=comfy_log, stderr=subprocess.STDOUT)
for _ in range(180):
    try:
        urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2); break
    except Exception:
        if comfy.poll() is not None:
            raise SystemExit("O ComfyUI encerrou. Veja " + f"{WORK}/comfyui.log")
        time.sleep(2)
else:
    raise SystemExit("O ComfyUI não respondeu em 6 minutos. Veja " + f"{WORK}/comfyui.log")
print("ComfyUI no ar.")

In [ ]:
# 5 · Proxy com token (cópia de workers/ltx-notebook/conveyer_proxy.py)
PROXY_SOURCE = r'''"""Token-checking proxy between the public tunnel and the notebook's ComfyUI.

ComfyUI has no authentication and listens on 127.0.0.1 only. This proxy is the one
thing the Cloudflare tunnel exposes: every request needs `Authorization: Bearer <token>`,
and only the six routes Conveyer's animateLtxScene uses are forwarded — no /api,
no custom-node routes, no websocket, no file browsing.

Runs inside the notebook (embedded into conveyer_ltx_notebook.ipynb by build_notebook.py).
Uses aiohttp, which ComfyUI already installs.

Env: CONVEYER_TOKEN (required), COMFY_URL, PROXY_PORT, MODELS_DIR, MODELS_JSON,
CONVEYER_PROFILE ("full" = LTX 22B, "lite" = LTX-Video 2B for a 16 GB T4).
"""
import hmac
import json
import os
import re

from aiohttp import ClientSession, ClientTimeout, web

TOKEN = os.environ["CONVEYER_TOKEN"]
COMFY = os.environ.get("COMFY_URL", "http://127.0.0.1:8188").rstrip("/")
PORT = int(os.environ.get("PROXY_PORT", "8189"))
MODELS_DIR = os.environ.get("MODELS_DIR", "ComfyUI/models")
MODELS_JSON = os.environ.get("MODELS_JSON", "models.json")
PROFILE = "lite" if os.environ.get("CONVEYER_PROFILE") == "lite" else "full"
VERIFIED_MARKER = ".conveyer-sha256-ok"

PROMPT_ID = re.compile(r"^[a-zA-Z0-9_-]{1,100}$")
MAX_UPLOAD = 25 * 1024 * 1024
MAX_PROMPT = 2 * 1024 * 1024


def authorized(request: web.Request) -> bool:
    header = request.headers.get("Authorization", "")
    return header.startswith("Bearer ") and hmac.compare_digest(header[7:].encode(), TOKEN.encode())


def missing_models() -> list:
    with open(MODELS_JSON) as fh:
        files = json.load(fh)["files"]
    missing = []
    for item in files:
        target = os.path.join(MODELS_DIR, item["path"])
        marker = target + VERIFIED_MARKER
        if not os.path.isfile(target) or os.path.getsize(target) != item["bytes"] or not os.path.isfile(marker):
            missing.append(item["path"])
    return missing


async def health(request: web.Request, session: ClientSession) -> web.Response:
    body = {"ok": True, "comfy": False, "models_ready": False, "gpu": "", "vram_gb": 0,
            "sparse_attention": False, "missing_models": [], "profile": PROFILE}
    try:
        async with session.get(f"{COMFY}/system_stats", timeout=ClientTimeout(total=5)) as resp:
            stats = await resp.json()
        device = (stats.get("devices") or [{}])[0]
        body["comfy"] = True
        body["gpu"] = str(device.get("name", ""))[:120]
        body["vram_gb"] = round(float(device.get("vram_total", 0)) / 1024 ** 3, 1)
        async with session.get(f"{COMFY}/object_info/BlockSparseAttention", timeout=ClientTimeout(total=10)) as resp:
            body["sparse_attention"] = resp.status == 200 and "BlockSparseAttention" in (await resp.json())
    except Exception:
        pass
    try:
        body["missing_models"] = missing_models()
        body["models_ready"] = not body["missing_models"]
    except Exception:
        body["missing_models"] = ["models.json"]
    return web.json_response(body)


async def forward(request: web.Request, session: ClientSession, route: str, max_body: int = 0) -> web.StreamResponse:
    data = None
    headers = {}
    if request.method == "POST":
        if request.content_length is not None and request.content_length > max_body:
            return web.json_response({"error": "body too large"}, status=413)
        # read(n) returns one chunk, not n bytes — loop to EOF, capped.
        chunks, size = [], 0
        async for chunk in request.content.iter_chunked(1 << 16):
            size += len(chunk)
            if size > max_body:
                return web.json_response({"error": "body too large"}, status=413)
            chunks.append(chunk)
        data = b"".join(chunks)
        if request.headers.get("Content-Type"):
            headers["Content-Type"] = request.headers["Content-Type"]
    url = f"{COMFY}{route}"
    async with session.request(request.method, url, data=data, headers=headers, params=request.query,
                               timeout=ClientTimeout(total=600), allow_redirects=False) as upstream:
        response = web.StreamResponse(status=upstream.status)
        response.content_type = upstream.content_type
        await response.prepare(request)
        async for chunk in upstream.content.iter_chunked(1 << 16):
            await response.write(chunk)
        await response.write_eof()
        return response


def make_app() -> web.Application:
    app = web.Application(client_max_size=MAX_UPLOAD + 1024)

    async def startup(app_: web.Application):
        app_["session"] = ClientSession()

    async def cleanup(app_: web.Application):
        await app_["session"].close()

    app.on_startup.append(startup)
    app.on_cleanup.append(cleanup)

    @web.middleware
    async def require_token(request: web.Request, handler):
        if not authorized(request):
            return web.json_response({"error": "unauthorized"}, status=401)
        return await handler(request)

    app.middlewares.append(require_token)

    def s(request):
        return request.app["session"]

    async def history(request):
        prompt_id = request.match_info["prompt_id"]
        if not PROMPT_ID.match(prompt_id):
            return web.json_response({"error": "bad id"}, status=400)
        return await forward(request, s(request), f"/history/{prompt_id}")

    app.router.add_get("/conveyer/health", lambda r: health(r, s(r)))
    app.router.add_post("/upload/image", lambda r: forward(r, s(r), "/upload/image", MAX_UPLOAD))
    app.router.add_get("/queue", lambda r: forward(r, s(r), "/queue"))
    app.router.add_post("/prompt", lambda r: forward(r, s(r), "/prompt", MAX_PROMPT))
    app.router.add_get("/history/{prompt_id}", history)
    app.router.add_get("/view", lambda r: forward(r, s(r), "/view"))
    return app


if __name__ == "__main__":
    web.run_app(make_app(), host="127.0.0.1", port=PORT, print=None)
'''
with open(f"{WORK}/conveyer_proxy.py", "w") as fh:
    fh.write(PROXY_SOURCE)
CONVEYER_TOKEN = secrets.token_urlsafe(32)
proxy_env = dict(os.environ, CONVEYER_TOKEN=CONVEYER_TOKEN, CONVEYER_PROFILE=PROFILE, MODELS_DIR=f"{COMFY_DIR}/models",
                 MODELS_JSON=MODELS_JSON, PROXY_PORT="8189")
proxy = subprocess.Popen([sys.executable, f"{WORK}/conveyer_proxy.py"], env=proxy_env,
                         stdout=open(f"{WORK}/proxy.log", "a"), stderr=subprocess.STDOUT)
time.sleep(3)
if proxy.poll() is not None:
    raise SystemExit("O proxy encerrou. Veja " + f"{WORK}/proxy.log")
print("Proxy no ar em 127.0.0.1:8189")

In [ ]:
# 6 · Túnel Cloudflare (gratuito, sem conta) → mostra endereço e token
cf = f"{WORK}/cloudflared"
if not os.path.exists(cf):
    subprocess.run(["wget", "-q", "-O", cf,
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
    os.chmod(cf, 0o755)
tunnel_log = f"{WORK}/tunnel.log"
tunnel = subprocess.Popen([cf, "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8189"],
                          stdout=open(tunnel_log, "w"), stderr=subprocess.STDOUT)
TUNNEL_URL = None
for _ in range(60):
    found = re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", open(tunnel_log).read())
    if found:
        TUNNEL_URL = found[0]; break
    time.sleep(1)
if not TUNNEL_URL:
    raise SystemExit("O túnel não informou endereço. Veja " + tunnel_log)
print("\nCole no Conveyer (/tools/ltx-scene-test → GPU do Colab ou Kaggle):\n")
print("  Endereço:", TUNNEL_URL)
print("  Token:   ", CONVEYER_TOKEN)
print("\nO token muda a cada execução desta célula 5.")

In [ ]:
# 7 · Acompanhar (opcional): mostra o fim do log do ComfyUI a cada 30 s. Interrompa quando quiser.
while True:
    for name, proc in (("ComfyUI", comfy), ("proxy", proxy), ("túnel", tunnel)):
        if proc.poll() is not None:
            print(f"ATENÇÃO: {name} parou (código {proc.returncode}). Rode de novo as células 4–6.")
    print("".join(open(f"{WORK}/comfyui.log").readlines()[-8:]))
    time.sleep(30)